# Notebook 01c · Trích xuất Top 10 Gloss & Tải Dữ liệu Huấn Luyện (ASL-Citizen)

Notebook này thực hiện **Bước 1 trong Pipeline Machine Learning**:
1. **Tận dụng kết quả EDA toàn diện**: Nạp bảng thống kê và metadata đã phân tích từ Notebook 01b (`outputs/metrics/full_eda_bundle/`).
2. **Trích xuất Top 10 Gloss chất lượng cao nhất**:
   - Cân bằng mẫu chặt chẽ ở cả 3 split ($N_{train} \ge 12, N_{val} \ge 5, N_{test} \ge 10$).
   - Đa dạng người ký cao ($Signers_{train} \ge 12$) để mô hình học đặc trưng ký hiệu tổng quát, tránh học vẹt người ký.
   - Từ vựng sạch (`n_variants == 1`), không chứa hậu tố số, phù hợp nhận diện cử chỉ bàn tay.
   - Hỗ trợ ghi đè danh sách tùy chọn qua biến `OVERRIDE_TOP10`.
3. **Tải toàn bộ video của 10 gloss** (~311 video, ~182 MB) về thư mục `data/raw/videos/` qua cơ chế HTTP Range Zip download (idempotent, đa luồng, tự resume).
4. **Kiểm tra trực quan (Contact Sheet)**: Kiểm tra đọc video bằng OpenCV và vẽ ảnh lưới các mẫu cử chỉ.
5. **Tạo & Xác minh Splits chính thức**:
   - Sinh các file `train.csv`, `val.csv`, `test.csv`, `classes.csv` trong `data/splits/`.
   - Kiểm tra rò rỉ người ký (Signer Leakage Check: $Train \cap Val = \emptyset$, $Train \cap Test = \emptyset$).
   - Đảm bảo 100% video tồn tại trên đĩa, sẵn sàng cho **Notebook 02: Landmark Extraction**.

## 1. Setup môi trường & Khởi tạo đường dẫn

Cài đặt tự động các gói phụ trợ nếu thiếu (hoạt động tốt trên cả Kaggle / Google Colab / Local).

In [ ]:
import os, sys, time, json, shutil, importlib, subprocess
from pathlib import Path

# Tự động cài đặt thư viện phụ trợ nếu thiếu (Kaggle/Colab)
for pkg in ["remotezip", "pyyaml", "tqdm", "opencv-python"]:
    try:
        importlib.import_module(pkg.replace("-", "_").replace("opencv_python", "cv2"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg], stdout=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Thêm repo root vào sys.path để import các module trong src/
cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd, *cwd.parents]:
    if (p / "configs" / "config.yaml").exists():
        repo_root = p
        break
if repo_root is None:
    repo_root = cwd

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import asl_citizen as A

P = A.resolve_paths(repo_root)
print(" Thư mục dự án:", repo_root)
print(" Thư mục video đích:", P.videos)
print(" Thư mục splits:", P.splits)
print(" Thư mục metrics/outputs:", P.metrics)

## 2. Nạp Bảng Thống Kê EDA & Metadata ASL-Citizen

Ưu tiên nạp từ cache bundle của đợt Full EDA trước (`outputs/metrics/full_eda_bundle/`). Nếu chạy trong môi trường mới toanh (như Kaggle session mới), hàm sẽ tự động trích xuất metadata từ máy chủ ASL Citizen qua HTTP Range.

In [ ]:
def load_eda_tables(P):
    # 1. Tìm trong các thư mục cache bundle
    search_dirs = [
        P.metrics / "full_eda_bundle",
        P.root / "outputs" / "metrics" / "full_eda_bundle",
        P.metrics,
        P.metrics / "full_eda_cache",
        P.manifests
    ]
    
    gloss_table_path, meta_path = None, None
    for d in search_dirs:
        if (d / "A2_gloss_table.csv").exists() and gloss_table_path is None:
            gloss_table_path = d / "A2_gloss_table.csv"
        if ((d / "meta_full.csv").exists() or (d / "meta_all.csv").exists()) and meta_path is None:
            meta_path = (d / "meta_full.csv") if (d / "meta_full.csv").exists() else (d / "meta_all.csv")
            
    if gloss_table_path and meta_path:
        print(f" Nạp thành công từ cache EDA local:")
        print(f" - Gloss table: {gloss_table_path}")
        print(f" - Metadata: {meta_path}")
        gt = pd.read_csv(gloss_table_path)
        meta = pd.read_csv(meta_path)
        return gt, meta
    
    # 2. Nếu thiếu file local, nạp qua HTTP Range zip của ASL Citizen
    print(" Không tìm thấy cache local, đang đọc metadata từ ASL Citizen...")
    zf = A.open_zip(A.DEFAULT_CFG["zip_src"])
    idx = A.index_zip(zf)
    cache_dir = P.metrics / "full_eda_cache"
    cache_dir.mkdir(parents=True, exist_ok=True)
    A.extract_small_files(zf, idx, cache_dir / "metadata")
    meta = A.load_metadata(cache_dir / "metadata", idx)
    gt = A.gloss_split_counts(meta)
    gt = gt.join(meta.groupby("gloss").signer_id.nunique().rename("signers_total"))
    gt = gt.join(meta[meta.split == "train"].groupby("gloss").signer_id.nunique().rename("signers_train"))
    gt["signers_train"] = gt["signers_train"].fillna(0).astype(int)
    meta["gloss_base"] = meta["gloss"].str.replace(r"\d+$", "", regex=True)
    gt = gt.join(meta.groupby("gloss").gloss_base.first())
    gt["n_variants"] = gt.groupby("gloss_base")["gloss_base"].transform("size")
    return gt, meta

gt, meta = load_eda_tables(P)
print(f"Tổng số gloss: {len(gt):,} | Tổng số video trong metadata: {len(meta):,}")

## 3. Lọc & Trích Xuất Top 10 Gloss Tốt Nhất

### Tiêu chuẩn chọn lọc Top 10:
1. **Cân bằng số mẫu trên cả 3 split**: $N_{train} \ge 12$, $N_{val} \ge 5$, $N_{test} \ge 10$.
2. **Đa dạng người ký**: $Signers_{train} \ge 12$ người ký khác nhau trong tập train.
3. **Tên từ vựng sạch (`n_variants == 1`)**: Từ đơn giản, không có hậu tố số (như `BOY`, `DEEP`, `MANY`, `MATH`, `PERFECT`, `OOPS`, `BOX`, `TRUE`, `TOILET`, `WEDDING`).

*Bạn có thể chỉ định danh sách 10 từ yêu thích tại biến `OVERRIDE_TOP10` bên dưới nếu muốn.*

In [ ]:
# Cho phép tự chọn danh sách nếu muốn (để trống [] để tự động xếp hạng tối ưu)
OVERRIDE_TOP10 = []

if OVERRIDE_TOP10:
    selected_glosses = sorted(OVERRIDE_TOP10)
    print(f"Sử dụng danh sách OVERRIDE: {selected_glosses}")
else:
    df_cand = gt.copy()
    for col in ["train", "val", "test", "total", "signers_train", "signers_total"]:
        if col in df_cand.columns:
            df_cand[col] = pd.to_numeric(df_cand[col])
            
    # Lọc từ đơn, không chứa chữ số, không chứa gạch dưới
    is_clean = (~df_cand["gloss"].str.contains(r"\d")) & (df_cand["n_variants"] == 1) & (~df_cand["gloss"].str.contains(r"[_\s]"))
    cand_clean = df_cand[is_clean].copy()
    
    # Tính điểm cân bằng: ưu tiên gloss có min(train, test) lớn và val đạt tối đa 6 signer
    cand_clean["score"] = cand_clean[["train", "test"]].min(axis=1) + (cand_clean["val"] >= 6).astype(int)
    
    # Sắp xếp xếp hạng
    cand_clean = cand_clean.sort_values(by=["score", "val", "train", "total", "signers_train"], ascending=False)
    selected_glosses = sorted(cand_clean.head(10)["gloss"].tolist())

print("=" * 60)
print(f"TOP 10 GLOSSES ĐƯỢC CHỌN ({len(selected_glosses)} classes):")
print(selected_glosses)
print("=" * 60)

# Bảng chi tiết 10 gloss
top10_summary = gt[gt["gloss"].isin(selected_glosses)].copy().sort_values("gloss").reset_index(drop=True)
display(top10_summary[["gloss", "train", "val", "test", "total", "signers_train", "signers_total"]])

# Metadata của các video thuộc 10 gloss này
target_meta = meta[meta["gloss"].isin(selected_glosses)].copy().sort_values(["split", "gloss", "signer_id", "video_id"]).reset_index(drop=True)
print(f"\nTổng số video cần tải: {len(target_meta):,} video")
print(f"Dung lượng nén ước tính: {target_meta['file_size'].sum() / 1e6:.2f} MB")
print("Phân bố số video theo split:")
display(target_meta.groupby("split").size().rename("videos").to_frame())

## 4. Tải Toàn Bộ Video Cho Top 10 Gloss (HTTP Range Download)

Tải trực tiếp các video của 10 class qua giao thức HTTP Range từ kho `ASL_Citizen.zip`:
- Chỉ tải đúng ~311 file video (~182 MB).
- Đa luồng (`workers=8`), tự động bỏ qua file đã tải đủ byte trên đĩa (resume an toàn).
- Ghi video vào thư mục `data/raw/videos/`.

In [ ]:
P.videos.mkdir(parents=True, exist_ok=True)
download_log_path = P.metrics / "top10_download_log.csv"

print(f" Bắt đầu tải video vào thư mục: {P.videos} ...")
t0 = time.time()

download_log = A.download_videos(
    src=A.DEFAULT_CFG["zip_src"],
    pool_df=target_meta[["zip_name", "file_size"]],
    out_dir=P.videos,
    workers=A.DEFAULT_CFG["workers"],
    log_path=download_log_path
)

elapsed = time.time() - t0
print(f" Hoàn thành tải trong {elapsed:.1f}s!")

# Xác minh tính toàn vẹn các file trên ổ cứng
target_meta["local_path"] = target_meta["video_file"].map(lambda f: str(P.videos / Path(f).name))
target_meta["download_ok"] = target_meta["local_path"].map(lambda p: Path(p).exists() and Path(p).stat().st_size > 0)

success_count = int(target_meta["download_ok"].sum())
total_count = len(target_meta)
print(f"\nKết quả kiểm tra file: {success_count}/{total_count} video hợp lệ trên đĩa.")

if success_count < total_count:
    failed = target_meta[~target_meta["download_ok"]]
    print(" CẢNH BÁO: Một số video bị lỗi tải. Hãy chạy lại cell này để thử lại:")
    display(failed[["video_id", "gloss", "split", "video_file"]].head(10))
else:
    print(" 100% video đã được tải thành công và sẵn sàng!")

## 5. Kiểm Tra Toàn Vẹn Video & Trực Quan Hóa (Contact Sheet)

Sử dụng OpenCV để đọc ngẫu nhiên một số video (đo FPS, độ phân giải, độ dài) và vẽ Contact Sheet chứa 3 khung hình mẫu cho mỗi gloss để kiểm tra trực quan chất lượng hình ảnh và cử chỉ.

In [ ]:
import cv2

# Đo thông số video của từng gloss
video_samples = []
for gloss in selected_glosses:
    sub = target_meta[(target_meta["gloss"] == gloss) & target_meta["download_ok"]]
    if len(sub) > 0:
        first_path = sub.iloc[0]["local_path"]
        stats = A.video_stats(first_path)
        video_samples.append({"gloss": gloss, **stats})

vdf = pd.DataFrame(video_samples)
display(vdf[["gloss", "ok", "fps", "width", "height", "duration", "n_frames"]])

# Vẽ Contact Sheet hiển thị 3 mẫu cho mỗi gloss
fig, axes = plt.subplots(len(selected_glosses), 3, figsize=(10, 2.2 * len(selected_glosses)), squeeze=False)

for i, gloss in enumerate(selected_glosses):
    sub = target_meta[(target_meta["gloss"] == gloss) & target_meta["download_ok"]]
    sample_rows = sub.head(3)
    for j in range(3):
        ax = axes[i][j]
        ax.axis("off")
        if j < len(sample_rows):
            vpath = sample_rows.iloc[j]["local_path"]
            cap = cv2.VideoCapture(vpath)
            mid_frame = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2, 0)
            cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
            ok, frame = cap.read()
            cap.release()
            if ok:
                ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if j == 0:
            ax.set_title(f"{i}: {gloss}", loc="left", fontsize=12, fontweight="bold")

plt.tight_layout()
P.plots.mkdir(parents=True, exist_ok=True)
contact_sheet_path = P.plots / "contact_sheet_top10.png"
fig.savefig(contact_sheet_path, dpi=120)
print(f" Đã lưu Contact Sheet tại: {contact_sheet_path}")
plt.show()

## 6. Ghi Split Files & Kiểm Tra Rò Rỉ (Data Leakage Verification)

Tạo các file chia tập chính thức:
- `data/splits/train.csv`
- `data/splits/val.csv`
- `data/splits/test.csv`
- `data/splits/classes.csv`

Kiểm tra nghiêm ngặt bằng assertion:
1. **Tách biệt hoàn toàn người ký**: Signer ở tập Train không được xuất hiện ở tập Val hay Test.
2. **Toàn vẹn tệp**: Mọi video trong split files đều tồn tại trên đĩa.
3. **Độ phủ nhãn**: 10 class đều có mẫu đầy đủ ở cả 3 tập.

In [ ]:
P.splits.mkdir(parents=True, exist_ok=True)

# 1. Tạo label mapping: 0 -> 9
label_map = {g: i for i, g in enumerate(selected_glosses)}
print("Bảng ánh xạ nhãn (Class Mapping):")
for g, idx in label_map.items():
    print(f"  Label {idx}: {g}")

# 2. Chuẩn bị bảng dữ liệu split
splits_df = target_meta.copy()
splits_df["label"] = splits_df["gloss"].map(label_map)
splits_df["video_file"] = splits_df["video_file"].map(lambda p: Path(p).name)

# 3. Ghi các file split chính thức
for s in ["train", "val", "test"]:
    part = splits_df[splits_df["split"] == s][["video_id", "gloss", "split", "signer_id", "video_file", "label"]].sort_values("video_id")
    out_csv = P.splits / f"{s}.csv"
    part.to_csv(out_csv, index=False)
    print(f" Đã lưu {out_csv} ({len(part)} video)")

# Lưu classes.csv và label_map.json
classes_df = pd.DataFrame([{"class_id": idx, "gloss": g} for g, idx in label_map.items()])
classes_df.to_csv(P.splits / "classes.csv", index=False)
(P.splits / "label_map.json").write_text(json.dumps(label_map, indent=2), encoding="utf-8")

# 4. KIỂM TRA RÒ RỈ & TÍNH TOÀN VẸN (Assertions)
train_df = pd.read_csv(P.splits / "train.csv")
val_df = pd.read_csv(P.splits / "val.csv")
test_df = pd.read_csv(P.splits / "test.csv")

train_signers = set(train_df["signer_id"])
val_signers = set(val_df["signer_id"])
test_signers = set(test_df["signer_id"])

# Kiểm tra rò rỉ signer
assert train_signers.isdisjoint(val_signers), " LỖI: Rò rỉ signer giữa train và val!"
assert train_signers.isdisjoint(test_signers), " LỖI: Rò rỉ signer giữa train và test!"
assert val_signers.isdisjoint(test_signers), " LỖI: Rò rỉ signer giữa val và test!"
print("\n KIỂM TRA ĐẠT: Tuyệt đối không có rò rỉ signer giữa train/val/test!")

# Kiểm tra file tồn tại trên đĩa
all_vids = pd.concat([train_df, val_df, test_df])["video_file"].tolist()
missing = [f for f in all_vids if not (P.videos / f).exists()]
assert len(missing) == 0, f" LỖI: Thiếu {len(missing)} file video trên đĩa!"
print(" KIỂM TRA ĐẠT: Toàn bộ video trong split files đều tồn tại trên đĩa!")

# Kiểm tra phân bố class đồng đều
assert set(train_df["gloss"]) == set(val_df["gloss"]) == set(test_df["gloss"]), " LỖI: Thiếu class ở một split!"
print(" KIỂM TRA ĐẠT: 10/10 class đều có mẫu đầy đủ ở cả 3 split!")

# Biểu đồ phân bố
balance_pivot = pd.concat([train_df, val_df, test_df]).groupby(["gloss", "split"]).size().unstack(fill_value=0)[["train", "val", "test"]]
display(balance_pivot)

fig, ax = plt.subplots(figsize=(10, 4))
balance_pivot.plot.bar(stacked=True, ax=ax, title="Phân bố số mẫu của 10 Gloss theo Split")
plt.ylabel("Số lượng video")
plt.tight_layout()
fig.savefig(P.plots / "top10_balance.png", dpi=120)
plt.show()

## 7. Cập Nhật Cấu Hình & Bàn Giao Dữ Liệu

Ghi file báo cáo tổng hợp `outputs/metrics/top10_selection_report.json` và cập nhật cấu hình `configs/config.yaml` (`num_classes: 10`).

Dữ liệu huấn luyện đã sẵn sàng! Bạn có thể chuyển sang **`notebooks/02_landmark_extraction.ipynb`** để trích xuất 126 đặc trưng MediaPipe Hands.

In [ ]:
import yaml

# Cập nhật configs/config.yaml
cfg_path = repo_root / "configs" / "config.yaml"
if cfg_path.exists():
    try:
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg_dict = yaml.safe_load(f) or {}
        if "dataset" not in cfg_dict:
            cfg_dict["dataset"] = {}
        cfg_dict["dataset"]["name"] = "ASL-Citizen"
        cfg_dict["dataset"]["num_classes"] = len(selected_glosses)
        with open(cfg_path, "w", encoding="utf-8") as f:
            yaml.dump(cfg_dict, f, default_flow_style=False, sort_keys=False)
        print(f" Đã cập nhật {cfg_path} -> num_classes: {len(selected_glosses)}")
    except Exception as e:
        print(f"Lưu ý: Không thể ghi đè YAML: {e}")

# Ghi báo cáo tóm tắt ra JSON
report = {
    "num_classes": len(selected_glosses),
    "classes": selected_glosses,
    "total_videos": len(all_vids),
    "splits_breakdown": {
        "train": len(train_df),
        "val": len(val_df),
        "test": len(test_df)
    },
    "signers_breakdown": {
        "train": len(train_signers),
        "val": len(val_signers),
        "test": len(test_signers)
    },
    "generated_at": time.strftime("%Y-%m-%d %H:%M:%S")
}

report_path = P.metrics / "top10_selection_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f" Đã ghi báo cáo tổng hợp: {report_path}")

print("\n" + "=" * 60)
print(" BƯỚC 1 HOÀN TẤT! Sẵn sàng chuyển sang Notebook 02: Landmark Extraction")
print("=" * 60)